# 02 - Baseline: TF-IDF + Regresión Logística
Prueba inicial con un modelo clásico para tener una referencia base de métricas y tiempo antes de pasar a un Transformer.

In [1]:
import time
import joblib
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score

## Carga de datos
Leo `train.csv` y `test.csv` que ya quedaron limpios y separados en el paso anterior.

In [4]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train["Comment"] = train["Comment"].fillna("")
test["Comment"] = test["Comment"].fillna("")

X_train_text = train["Comment"]
y_train = train["Sentiment"]

X_test_text = test["Comment"]
y_test = test["Sentiment"]

## Vectorización con TF-IDF
Paso los comentarios a números con TF-IDF usando unigramas y bigramas (`ngram_range=(1, 2)`) y un máximo de 5000 términos.

In [5]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test:", X_test.shape)

Dimensiones de X_train: (166756, 5000)
Dimensiones de X_test: (41690, 5000)


## Entrenamiento
Entreno la regresión logística y mido cuánto tarda en ajustar el modelo.

In [6]:
model = LogisticRegression(max_iter=1000, random_state=42)
t0_train = time.perf_counter()
model.fit(X_train, y_train)
train_time = time.perf_counter() - t0_train

## Evaluación y tiempo de predicción
Reviso las métricas en `test` (sobre todo el Macro F1) y cuánto se demora en predecir los comentarios.

In [7]:
t0_pred = time.perf_counter()
y_pred = model.predict(X_test)
infer_time = time.perf_counter() - t0_pred

target_names = ["negativo (0)", "neutral (1)", "positivo (2)"]
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=target_names))

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
ms_per_sample = (infer_time / len(X_test_text)) * 1000


Reporte de Clasificación:
              precision    recall  f1-score   support

negativo (0)       0.79      0.66      0.72      9485
 neutral (1)       0.74      0.85      0.79     13638
positivo (2)       0.84      0.83      0.84     18567

    accuracy                           0.80     41690
   macro avg       0.79      0.78      0.78     41690
weighted avg       0.80      0.80      0.80     41690



In [8]:
print(f"Accuracy: {acc:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}")
print(f"Tiempo de entrenamiento: {train_time:.2f} s")
print(f"Tiempo de inferencia total ({len(X_test_text)} muestras): {infer_time:.4f} s ({ms_per_sample:.4f} ms/comentario)")

Accuracy: 0.7964
Macro F1-Score: 0.7825
Tiempo de entrenamiento: 19.26 s
Tiempo de inferencia total (41690 muestras): 0.0706 s (0.0017 ms/comentario)


## Guardar modelo y notas
Guardo el vectorizador y el modelo para no tener que reentrenarlos después.
- Me dió ~0.80 de accuracy y ~0.78 de macro F1, y predice rápido.
- En `negativo (0)` le cuesta un poco más el recall (0.66), así que habrá que ver cómo responde el Transformer y qué sale en los tópicos de negativos.

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump(vectorizer, "models/tfidf_vectorizer.joblib")
joblib.dump(model, "models/logreg_baseline.joblib")
print("\nArtefactos guardados en la carpeta 'models/'.")

NameError: name 'Path' is not defined